# MLlib

Spark biedt ook een framework aan voor MachineLearning modellen te trainen op gedistribueerde datasets.
Dit framework is MLlib of ook wel sparkML genoemd.
De code om te werken met deze package is sterk gelijkaardig aan sklearn.
De API en een uitgebreide documentatie met voorbeeldcode kan je [hier](https://spark.apache.org/docs/latest/ml-guide.html) vinden.

Deze package bied de volgende tools aan
* ML-technieken: classificatie, regressie, clustering, ...
* Features: Extracting en transforming van features, PCA, ...
* Pipelines: Maak, train, optimaliseer en evalueer pipelines
* Persistentie: Bewaar en laden van algoritmes/modellen
* Databeheer: Algebra tools, statistieken, null-waarden, ...

Let op dat er twee API's aangeboden worden, 1 gebaseerd op RDD's en 1 op DataFrames.
De API gebaseerd op RDD's is ouder en minder flexibel dan de API gebruik makend van DataFrames.
Momenteel werken ze allebei maar in de toekomst zou de RDD gebaseerde kunnen verdwijnen.

## Utilities

### Varianten voor numpy-arrays

Voor feature sets en volledige matrices van datasets aan te maken kan je gebruik maken van de Vector en Matrix klassen.
Deze beschikken over een Dense variant waar je elk element moet ingeven of een Sparse Variant waar cellen, elementen leeg kan laten.
Dit ziet er als volgt uit:

In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.linalg import Vectors
from pyspark.ml.linalg import Matrices

spark = SparkSession.builder \
    .appName("SparkMLlibDemo") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.301") \
    .config("spark.hadoop.fs.s3a.access.key", "bigdata") \
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

sc = spark.sparkContext

print(Vectors.sparse(4, [(0, 1.0), (3, -2.0)])) # lijst met 4 elementen, element 0 heeft waarde 1 en element 3 waarde -2
print(Vectors.dense([1.0, 0, 0, -2.0]))
print(Matrices.dense(4,4, range(16))) # matrix met 4 rijen en 4 kolommen, 0 tot 16 zijn de values

(4,[0,3],[1.0,-2.0])
[1.0,0.0,0.0,-2.0]
DenseMatrix([[ 0.,  4.,  8., 12.],
             [ 1.,  5.,  9., 13.],
             [ 2.,  6., 10., 14.],
             [ 3.,  7., 11., 15.]])


Het is belangrijk om te weten dat dit locale datastructuren (wrapper rond numpy array) zijn en geen gedistribueerde objecten.

### Statistieken

Voor er kan gewerkt worden met statistieken moeten we (net zoals bij pandas) eerst een dataset hebben.
Hieronder maken we een random dataframe aan van 50 rijen en 4 kolommen.

In [9]:
from pyspark.sql.functions import rand
n = 50 # rijen
d = 4 # kolommen

# Start from range(n) and add d random columns
df = spark.range(0, n)
for i in range(d):
    df = df.withColumn(f"c{i}", rand(seed=42 + i)) # invullen met random data

df.show()

#df.summary().show()
df.select([f"c{i}" for i in range(4)]).summary().show() # hiermee hou je enkel de c0 tot c3 kolommen over

+---+-------------------+-------------------+-------------------+--------------------+
| id|                 c0|                 c1|                 c2|                  c3|
+---+-------------------+-------------------+-------------------+--------------------+
|  0|  0.619189370225301| 0.8017532427858894| 0.7985334687187781|  0.7971301351894658|
|  1| 0.5096018842446481| 0.6565552949992319| 0.8679617292463396|  0.2422600602366628|
|  2| 0.8325259388871524| 0.2515595782593636| 0.5068407664758031|  0.7097364852287148|
|  3|0.26322809041172357| 0.2073428376111074| 0.7946993123364816|  0.4509378549789149|
|  4| 0.6702867696264135| 0.6392921379278927| 0.7463770360606422| 0.30357970527906064|
|  5| 0.5173283545794627| 0.8505582285081454|0.27140710651938027|  0.5794047153077633|
|  6| 0.9991441647585968| 0.8184715436384177| 0.2741303337234684|  0.7800738662272417|
|  7|0.06993233728279169| 0.7555506990689408| 0.8373499476137246|0.057394743608470966|
|  8| 0.9696695610826327|0.3438046953870188

**Correlation matrix**

Buiten de statistieken die berekend kunnen worden door de summary() functie kan ook de correlatiematrix belangrijk zijn.
Deze matrix maakt het mogelijk om het verband tussen de verscheidene features te bestuderen.
Deze matrix kan als volgt berekend worden voor een gedistribueerd dataframe.

In [15]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
df = df.select([f"c{i}" for i in range(4)]) # verwijder de id-kolom

# Eerst moeten we een kolom maken dat alle numerieke waarden groepeert in een vector
assembler = VectorAssembler(inputCols=df.columns, outputCol='vector')
df_vector = assembler.transform(df)

# op basis van dit df kan de correlatie matrix berekend worden
Correlation.corr(df_vector, 'vector').show(truncate=False) # heel vaak bij transformers moet je zeggen op welke kolom het moet uitgevoerd worden

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|pearson(vector)                                                                                                                                                                                                                                                                                                                                                       |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Onafhankelijksheidtest**

Naast de correlatiematrix kan het ook belangrijk zijn om de onafhankelijkheid te testen tussen elke feature en een label.
Dit kan uitgevoerd worden door een zogenaamde ChiSquareTest.
Deze krijgt als input een dataframe, de naam van de kolom met de features (als vectors) en de naam van een kolom met de labels.
We kunnen deze test uitvoeren als volgt:

In [22]:
from pyspark.sql.functions import when, rand
from pyspark.ml.stat import ChiSquareTest
df_ind = df_vector.withColumn('label', when(rand() > 0.5, 1).otherwise(0)) # bereken random labels 0 en 1
df_ind.show()

# voeg een label-kolom toe aan df_vector
ChiSquareTest.test(df_ind, 'vector', 'label', flatten=True).show()

+-------------------+-------------------+-------------------+--------------------+--------------------+-----+
|                 c0|                 c1|                 c2|                  c3|              vector|label|
+-------------------+-------------------+-------------------+--------------------+--------------------+-----+
|  0.619189370225301| 0.8017532427858894| 0.7985334687187781|  0.7971301351894658|[0.61918937022530...|    1|
| 0.5096018842446481| 0.6565552949992319| 0.8679617292463396|  0.2422600602366628|[0.50960188424464...|    0|
| 0.8325259388871524| 0.2515595782593636| 0.5068407664758031|  0.7097364852287148|[0.83252593888715...|    1|
|0.26322809041172357| 0.2073428376111074| 0.7946993123364816|  0.4509378549789149|[0.26322809041172...|    0|
| 0.6702867696264135| 0.6392921379278927| 0.7463770360606422| 0.30357970527906064|[0.67028676962641...|    0|
| 0.5173283545794627| 0.8505582285081454|0.27140710651938027|  0.5794047153077633|[0.51732835457946...|    0|
| 0.999144

De Chi-square test wordt op de volgende manieren gebruikt in Machine Learning:
* Feature selectie: Identificeer welke features een significante relatie hebben met de targetvariabele.
* Correlatie tussen categorische variabelen: Evalueer de afhankelijkheid tussen twee categorische variabelen.

De Chi-square test vergelijkt de verwachte en werkelijke frequenties van waarden in de categorieën van een feature ten opzichte van de targetvariabele. Het doel is om te bepalen of de verschillen tussen deze frequenties statistisch significant zijn of niet. Een hoge Chi-square score betekent dat er een grote afhankelijkheid is tussen de feature en de target, wat suggereert dat de feature nuttig is voor voorspellingen.
Als de p-waarde kleiner is dan een vooraf bepaald significantieniveau (bijvoorbeeld 0,05), verwerp je de nullhypothese (dat er geen relatie is), wat betekent dat de feature relevant is.

**Summarizer**

Andere statistieken per kolom kunnen berekend worden door gebruik te maken van de Summarizer klasse:

In [26]:
from pyspark.ml.stat import Summarizer

summarizer = Summarizer.metrics('mean', 'std', 'max')
df_vector.select(summarizer.summary(df_vector.vector)).show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|aggregate_metrics(vector, 1.0)                                                                                                                                                                                                                  |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{[0.5929594659966628,0.5392205982358973,0.4722234723982675,0.46231880514293267], [0.2733627501627969,0.27960181781580584,0.2820390630219527,0.28878456904991756], [0.9991441647585968,0.9688522184233442,0.9672886812333024,0.9874791218710566]}|
+---------------------------

Het gebruik maken van de Summarizer maakt het dus mogelijk om rechtstreeks op de feature vectors te werken zonder ze eerst terug te moeten splitsen.

### Pipelines

Pipelines binnen Spark zijn een groep van high-level API's steunend op Dataframes om ML-pipelines aan te maken, optimaliseren en trainen.
De belangrijkste concepten binnen de Pipelines van Spark zijn:
* Dataframe: concept van de dataset
* Transformer: Zet een dataframe om in een ander dataframe
* Estimator: Zet een dataframe om in een model/transformer
* Pipeline: een ketting van transformers en estimators om een flow vast te leggen
* Parameter: API voor parameters van transformers en estimators aan te passen

Gebruik nu onderstaande mini-dataset waar we op basis van een tekstkolom met logistische regressie een bepaald label proberen te voorspellen.
Maak hiervoor een Pipeline uit die bestaat uit de volgende stappen:
* Tokenizer om de tekstkolom te splitsen in de overeenkomstige woorden
* HashingTf om de term frequency van de woorden te bepalen en het om te zetten naar een feature vector
* LogisticRegression Estimator om de voorspelling te doen.

Train daarna deze pipeline en maak de voorspellingen voor de traningsdata.
Hoe accuraat is dit model?

In [27]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import HashingTF, Tokenizer

# Prepare training documents from a list of (id, text, label) tuples.
training = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0)
], ["id", "text", "label"])

tokenizer = Tokenizer(inputCol='text', outputCol='tokens')
hasher = HashingTF(inputCol='tokens', outputCol='features')
clf = LogisticRegression(maxIter=10, regParam=0.001) # default inputCols = features, default labelCol = 'label'
pipeline = Pipeline(stages=[tokenizer, hasher, clf])

model = pipeline.fit(training)
predictions = model.transform(training)
predictions.show()

26/03/17 14:02:51 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


+---+----------------+-----+--------------------+--------------------+--------------------+--------------------+----------+
| id|            text|label|              tokens|            features|       rawPrediction|         probability|prediction|
+---+----------------+-----+--------------------+--------------------+--------------------+--------------------+----------+
|  0| a b c d e spark|  1.0|[a, b, c, d, e, s...|(262144,[74920,89...|[-5.9388192690226...|[0.00262821349694...|       1.0|
|  1|             b d|  0.0|              [b, d]|(262144,[89530,14...|[5.62050636899131...|[0.99639027118011...|       0.0|
|  2|     spark f g h|  1.0|    [spark, f, g, h]|(262144,[36803,17...|[-6.1134100250082...|[0.00220810505702...|       1.0|
|  3|hadoop mapreduce|  0.0| [hadoop, mapreduce]|(262144,[132966,1...|[6.66214714866363...|[0.99872323370637...|       0.0|
+---+----------------+-----+--------------------+--------------------+--------------------+--------------------+----------+



### Evalueren van een model

In de pyspark.ml package zitten er ook functionaliteiten voor deze modellen te evalueren.
Meer informatie hierover vind je [hier](https://spark.apache.org/docs/2.2.0/mllib-evaluation-metrics.html).

In [30]:
# evalueren van het model
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
evaluator.evaluate(predictions)

1.0

### Data sources

Door gebruik te maken van de sparkContext kunnen een reeks standaard databronnen ingelezen worden om datasets uit op te bouwen (Csv, Json, ...).
Daarnaast is het ook mogelijk om een folder met een reeks beelden te gebruiken als dataset om zo een model voor image classification te trainen.
Download nu [deze](https://www.kaggle.com/returnofsputnik/chihuahua-or-muffin) dataset en upload ze naar een folder op het hadoop filesysteem.

In [31]:
import kagglehub
from minio import Minio
import os

path = kagglehub.dataset_download("returnofsputnik/chihuahua-or-muffin")

client = Minio(
    "minio1:9000",
    access_key="bigdata",
    secret_key="bigdata123",
    secure=False
)

bucket_name = "03-mllib"

# Make bucket if it doesn't exist
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

# Upload all images in the folder
for root, dirs, files in os.walk(path):
    for file in files:
        file_path = os.path.join(root, file)
        object_name = file  # You can include subfolders by using relative path if you want
        client.fput_object(bucket_name, f"chimuffin/{object_name}", file_path)
        print(f"Uploaded: {object_name}")

100%|██████████| 183k/183k [00:00<00:00, 548kB/s]

Extracting files...


Uploaded: chihuahua-4.jpg
Uploaded: muffin-7.jpeg
Uploaded: muffin-2.jpeg
Uploaded: chihuahua-1.jpg
Uploaded: muffin-3.jpeg
Uploaded: muffin-1.jpeg
Uploaded: chihuahua-6.jpg
Uploaded: chihuahua-8.jpg
Uploaded: chihuahua-7.jpg
Uploaded: chihuahua-5.jpg
Uploaded: chihuahua-3.jpg
Uploaded: muffin-5.jpeg
Uploaded: muffin-8.jpeg
Uploaded: muffin-4.jpeg
Uploaded: muffin-6.jpeg
Uploaded: chihuahua-2.jpg


De geuploade images kunnen nu ingelezen worden als volgt:

In [33]:
df = spark.read.format('image').option('dropInvalid', True).load('s3a://03-mllib/chimuffin')

df.printSchema()

df.select('image.origin', 'image.width', 'image.height').show(truncate=False)

root
 |-- image: struct (nullable = true)
 |    |-- origin: string (nullable = true)
 |    |-- height: integer (nullable = true)
 |    |-- width: integer (nullable = true)
 |    |-- nChannels: integer (nullable = true)
 |    |-- mode: integer (nullable = true)
 |    |-- data: binary (nullable = true)



[Stage 79:>                                                         (0 + 2) / 2]

+----------------------------------------+-----+------+
|origin                                  |width|height|
+----------------------------------------+-----+------+
|s3a://03-mllib/chimuffin/muffin-4.jpeg  |172  |170   |
|s3a://03-mllib/chimuffin/muffin-7.jpeg  |171  |172   |
|s3a://03-mllib/chimuffin/muffin-1.jpeg  |171  |171   |
|s3a://03-mllib/chimuffin/muffin-8.jpeg  |172  |172   |
|s3a://03-mllib/chimuffin/chihuahua-6.jpg|172  |169   |
|s3a://03-mllib/chimuffin/chihuahua-8.jpg|168  |172   |
|s3a://03-mllib/chimuffin/muffin-6.jpeg  |168  |169   |
|s3a://03-mllib/chimuffin/chihuahua-5.jpg|171  |169   |
|s3a://03-mllib/chimuffin/muffin-2.jpeg  |168  |171   |
|s3a://03-mllib/chimuffin/muffin-5.jpeg  |171  |169   |
|s3a://03-mllib/chimuffin/muffin-3.jpeg  |171  |170   |
|s3a://03-mllib/chimuffin/chihuahua-3.jpg|171  |170   |
|s3a://03-mllib/chimuffin/chihuahua-4.jpg|168  |170   |
|s3a://03-mllib/chimuffin/chihuahua-7.jpg|171  |172   |
|s3a://03-mllib/chimuffin/chihuahua-1.jpg|171  |

Merk op dat het werken met images niet zo eenvoudig is.
Hiervoor wordt binnen pyspark typisch gebruik gemaakt van de [sparkdl](https://smurching.github.io/spark-deep-learning/site/api/python/sparkdl.html) package.
Hierbij staat de dl voor deep learning.
Aangezien dit ons momenteel te ver leidt ga ik dit niet verder toelichten.

Een andere aparte databron die eenvoudig ingelezen kan worden is het formaat "libsvm".
Een bestand van dit formaat wordt ingelezen als een dataframe met twee kolommen: een label en een kolom met de feature-vectors.
De code om dergelijk bestand in te laden is:

In [ ]:
#df = spark.read.format("libsvm").load("{path to file here}")

## Voorbeeld

Train en evalueer een machine learning model dat beeldgegevens kan classificeren, gebaseerd op eigenschappen zoals hoogte, breedte en aantal kanalen. Omdat MLlib geen directe ondersteuning biedt voor beelddata, zullen we enkele numerieke eigenschappen van de afbeeldingen gebruiken en pipelines opzetten voor preprocessing en training.
Hierbij ga je verder werken op het dataframe van de chichuahua of muffin dataset van hierboven.

Voer de volgende stappen uit:
* Data decoderen en omzetten naar een dataframe dat door ML-kan gebruikt worden.
    * Schrijf een udf om de image.data kolom om te zetten naar een vector. Gebruik hiervoor een udf dat dit eerst omzet naar een numpy array waarna het omgezet wordt naar een PIL-Image (een python package). Dit wordt dan geresized naar een 32x32 figuur. Ten slotte wordt dit omgezet naar een 1D-array 
    * Schrijf een udf om de filename in de image.origin kolom om te zetten naar 1 van de labels: 'muffin' of 'chihuahua'
* Na het uitvoeren van de udf's zou je een dataframe moeten hebben met minstens 2 kolommen: het label als string kolom, de data als een vector kolom
* Splits de data in een trainings en testset
* Voer de volgende preprocessing stappen uit
    * Zet de tekstuele labels om naar integers
    * Schaal de vector-kolom door elke waarde te delen door 255 zodat de pixel-waarden in de range 0-255 vallen
    * Zorg ervoor dat dit in een pipeline zit en fit de preprocessor al eens. Print het schema en een aantal rijen uit om de output te verifieren
* Voer data modelling uit
    * Zorg voor dimensionality reduction aan de hand van PCA
    * Gebruik logistische regressie om het label te voorspellen
* Evalueer het model door een confusion matrix te berekenen

In [41]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, StringType
from pyspark.ml.feature import StringIndexer, MinMaxScaler, PCA
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.functions import array_to_vector
from PIL import Image
import io
import numpy as np

# udf - user defined function - python functie die rij per rij uitgevoerd wordt (zoals de apply van pandas)
def get_label(filename):
    if 'chihuahua' in filename:
        return 'chihuahua'
    else:
        return 'muffin'

get_label_udf = F.udf(get_label, StringType()) # param 1 is de python functie die uitgevoerd wordt, param 2 is het returntype
df_data = df.withColumn('label', get_label_udf(F.col('image.origin')))

def resize_image(pixel_data, height, width, target_size=(32,32)): # we willen de figuren resizen naar 32 bij 32 pixels
    # ik verwacht niet dat je dit kan in evaluaties
    image_array = np.array(pixel_data, dtype=np.uint8).reshape((height, width, 3)) # maak van de vector pixel_data een figuur
    image = Image.fromarray(image_array, mode='RGB') # numpy array to PIL image
    image = image.resize(target_size) # resize de figuur
    image = np.array(image, dtype=float).flatten().tolist() # terug omzetten naar 1d via flatten, flatten doet 2d -> 1d
    return image
    
resize_image_udf = F.udf(resize_image, ArrayType(DoubleType())) # param 1 is de python functie die uitgevoerd wordt, param 2 is het returntype
df_data = df_data.withColumn('features', array_to_vector(resize_image_udf(F.col('image.data'), F.col('image.height'), F.col('image.width'))))

df_data.select('image.origin', 'label', 'features').show()

+--------------------+---------+--------------------+
|              origin|    label|            features|
+--------------------+---------+--------------------+
|s3a://03-mllib/ch...|   muffin|[151.0,119.0,119....|
|s3a://03-mllib/ch...|   muffin|[254.0,254.0,254....|
|s3a://03-mllib/ch...|   muffin|[27.0,28.0,34.0,2...|
|s3a://03-mllib/ch...|   muffin|[24.0,99.0,155.0,...|
|s3a://03-mllib/ch...|chihuahua|[193.0,174.0,168....|
|s3a://03-mllib/ch...|chihuahua|[168.0,159.0,177....|
|s3a://03-mllib/ch...|   muffin|[204.0,191.0,223....|
|s3a://03-mllib/ch...|chihuahua|[107.0,98.0,168.0...|
|s3a://03-mllib/ch...|   muffin|[38.0,58.0,97.0,3...|
|s3a://03-mllib/ch...|   muffin|[204.0,212.0,228....|
|s3a://03-mllib/ch...|   muffin|[211.0,220.0,228....|
|s3a://03-mllib/ch...|chihuahua|[89.0,86.0,121.0,...|
|s3a://03-mllib/ch...|chihuahua|[15.0,18.0,24.0,2...|
|s3a://03-mllib/ch...|chihuahua|[9.0,26.0,112.0,1...|
|s3a://03-mllib/ch...|chihuahua|[25.0,72.0,128.0,...|
|s3a://03-mllib/ch...|chihua

In [44]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import PipelineModel

train_data, test_data = df_data.randomSplit([0.8, 0.2], seed=42)
train_data.show(5)

label_indexer = StringIndexer(inputCol="label", outputCol="label_index")
scaler = MinMaxScaler(inputCol="features", outputCol="scaled_features")
preprocessor = Pipeline(stages=[label_indexer, scaler])

preprocessor_model =  preprocessor.fit(train_data)
train_data_preprocessed = preprocessor_model.transform(train_data)
test_data_preprocessed = preprocessor_model.transform(test_data)

train_data_preprocessed.show(5)

# remove the data.col_x so the vector assembler can take it
train_data_preprocessed.show(5)
train_data_preprocessed.printSchema()

# modelling
pca = PCA(k=50, inputCol="scaled_features", outputCol="pca_features")
lr = LogisticRegression(featuresCol="pca_features", labelCol="label_index", maxIter=10)
pipeline = Pipeline(stages=[preprocessor, pca, lr])

# Train het model
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

# Evalueer het model
confusion_df = (
    predictions
    .groupBy("label_index")
    .pivot("prediction")
    .count()
    .fillna(0)
    .orderBy("label_index")
)

confusion_df.show()

evaluator = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

+--------------------+---------+--------------------+
|               image|    label|            features|
+--------------------+---------+--------------------+
|{s3a://03-mllib/c...|chihuahua|[193.0,174.0,168....|
|{s3a://03-mllib/c...|chihuahua|[168.0,159.0,177....|
|{s3a://03-mllib/c...|   muffin|[151.0,119.0,119....|
|{s3a://03-mllib/c...|   muffin|[254.0,254.0,254....|
|{s3a://03-mllib/c...|   muffin|[24.0,99.0,155.0,...|
+--------------------+---------+--------------------+
only showing top 5 rows



+--------------------+---------+--------------------+-----------+--------------------+
|               image|    label|            features|label_index|     scaled_features|
+--------------------+---------+--------------------+-----------+--------------------+
|{s3a://03-mllib/c...|chihuahua|[193.0,174.0,168....|        0.0|[0.75102040816326...|
|{s3a://03-mllib/c...|chihuahua|[168.0,159.0,177....|        0.0|[0.64897959183673...|
|{s3a://03-mllib/c...|   muffin|[151.0,119.0,119....|        1.0|[0.57959183673469...|
|{s3a://03-mllib/c...|   muffin|[254.0,254.0,254....|        1.0|[1.0,1.0,1.0,1.0,...|
|{s3a://03-mllib/c...|   muffin|[24.0,99.0,155.0,...|        1.0|[0.06122448979591...|
+--------------------+---------+--------------------+-----------+--------------------+
only showing top 5 rows

+--------------------+---------+--------------------+-----------+--------------------+
|               image|    label|            features|label_index|     scaled_features|
+-----------------

26/03/17 15:20:17 WARN DAGScheduler: Broadcasting large task binary with size 1355.2 KiB
26/03/17 15:20:17 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WARN DAGScheduler: Broadcasting large task binary with size 1355.9 KiB
26/03/17 15:20:18 WAR

+-----------+---+
|label_index|0.0|
+-----------+---+
|        0.0|  2|
|        1.0|  2|
+-----------+---+

Accuracy: 0.5


In [45]:
model.write().overwrite().save("s3a://03-mllib/pipeline")

loaded_model = PipelineModel.load("s3a://03-mllib/pipeline")

26/03/17 15:20:25 WARN TaskSetManager: Stage 212 contains a task of very large size (1233 KiB). The maximum recommended task size is 1000 KiB.


## Oefening - classificatie

Voer de volgende stappen uit, volg de best-practices van het data science vak:
* Genereer een dataset met de make_classification functie van Scikit-Learn (of gebruik randomSplit op een bestaande dataset in PySpark).
* Plaats de dataset op de MinIO cluster
* Converteer de dataset naar een PySpark DataFrame door het te lezen vanaf de MinIO cluster.
* Voer basisverkenning uit: toon het schema van de data, bekijk de eerste rijen, en controleer de kolomtypes.
* Gebruik VectorAssembler om alle features in één vector-kolom (features) te combineren.
* Controleer de structuur van de output.
* Train een Logistic Regression model op de gegenereerde dummy data.
* Gebruik een training/test split van 80/20.
* Evalueer het model door de nauwkeurigheid (accuracy) te berekenen.

## Oefening - hyperparameter tuning

Voer de volgende stappen uit, volg de best-practices van het data science vak:
* Genereer dummy data (of gebruik bestaande data) om een regressieprobleem op te lossen. (make_regression)
* Bouw een pipeline die de data voorbereidt door schaling uit te voeren, een RandomForest-model traint, en de beste hyperparameters selecteert. (Tip: ParamGridBuilder)
* Voer hyperparameter tuning uit en evalueer de prestaties van het beste model.